# 02 - Training and Evaluation

This notebook covers the training and evaluation phase of the Neural Machine Translation (NMT) project.
It loads the preprocessed data and vocabularies, initializes the sequence-to-sequence model (Encoder-Decoder with Attention), and runs the training loop using Teacher Forcing.

It also includes placeholders for optional tasks such as BLEU score evaluation and Beam Search decoding.

In [1]:
import os
import sys
import random
import math
import time

sys.path.append(os.path.abspath(os.path.join('..','src')))

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Import our custom modules
from vocabulary import Vocabulary
from dataset import TranslationDataset, collate_fn, load_processed
from encoder import Encoder
from decoder import Decoder
from seq2seq import Seq2Seq
from transformer import TransformerSeq2Seq

print('PyTorch version:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

PyTorch version: 2.8.0+cu128
Using device: cuda


## 1. Configuration & Hyperparameters
Define the sizes of the neural network layers, learning rate, and batch size. This is where you can perform hyperparameter tuning.

In [2]:
data_dir = os.path.abspath(os.path.join('..','data','processed'))
models_dir = os.path.abspath(os.path.join('..','models'))

os.makedirs(models_dir, exist_ok=True)

# Choose which model to train/evaluate: 'RNN' or 'TRANSFORMER'.
# Everything downstream (data loading, training loop, BLEU) is shared -- only
# the model-construction cell branches on this flag.
MODEL_TYPE = 'TRANSFORMER'

# --- Shared hyperparameters ---
BATCH_SIZE = 64
N_EPOCHS = 10
CLIP = 1.0

# --- RNN hyperparameters (used when MODEL_TYPE == 'RNN') ---
ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
ENC_HID_DIM = 256
DEC_HID_DIM = 256
ATTN_DIM = 256
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

# --- Transformer hyperparameters (used when MODEL_TYPE == 'TRANSFORMER') ---
D_MODEL = 256      # model/embedding width (must be divisible by N_HEADS)
N_HEADS = 8        # number of attention heads
N_LAYERS = 3       # how many encoder layers and decoder layers to stack
D_FF = 512         # hidden size of the feed-forward sub-layer
TF_DROPOUT = 0.1   # transformers use lighter dropout than the RNN above

# Transformers train better with a smaller Adam learning rate than the RNN.
LEARNING_RATE = 0.001 if MODEL_TYPE == 'RNN' else 0.0005

## 2. Loading Data
Load the `.pt` files and vocabularies created in the `01_data_preparation.ipynb` notebook.

In [3]:
# Load vocabularies
src_vocab = Vocabulary.load(os.path.join(data_dir, "vocab.src.json"))
tgt_vocab = Vocabulary.load(os.path.join(data_dir, "vocab.tgt.json"))

print(f"Source vocabulary size: {len(src_vocab)}")
print(f"Target vocabulary size: {len(tgt_vocab)}")

# Load datasets
train_data = load_processed(os.path.join(data_dir, "train.pt"))
val_data = load_processed(os.path.join(data_dir, "val.pt"))
test_data = load_processed(os.path.join(data_dir, "test.pt"))

print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(val_data)}")
print(f"Testing examples: {len(test_data)}")

pad_idx = src_vocab.token2id[Vocabulary.PAD]

# Create DataLoaders
train_loader = DataLoader(TranslationDataset(train_data), batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda b: collate_fn(b, pad_id=pad_idx))
val_loader = DataLoader(TranslationDataset(val_data), batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda b: collate_fn(b, pad_id=pad_idx))
test_loader = DataLoader(TranslationDataset(test_data), batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda b: collate_fn(b, pad_id=pad_idx))

Source vocabulary size: 29961
Target vocabulary size: 48596
Training examples: 247865
Validation examples: 13770
Testing examples: 13771


## 3. Model Initialization
Instantiate the Encoder, Decoder, and the full Seq2Seq model. We also define the Optimizer (Adam) and the Loss Function (CrossEntropyLoss).

In [4]:
INPUT_DIM = len(src_vocab)
OUTPUT_DIM = len(tgt_vocab)
SOS_IDX = tgt_vocab.token2id[Vocabulary.SOS]
EOS_IDX = tgt_vocab.token2id[Vocabulary.EOS]

if MODEL_TYPE == 'RNN':
    enc = Encoder(INPUT_DIM, ENC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, dropout=ENC_DROPOUT, pad_idx=pad_idx)
    if enc.bidirectional:
        enc_hid_dim = ENC_HID_DIM * 2
    else:
        enc_hid_dim = ENC_HID_DIM
    dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, enc_hid_dim, DEC_HID_DIM, ATTN_DIM, dropout=DEC_DROPOUT, pad_idx=pad_idx)
    model = Seq2Seq(enc, dec, device, pad_idx, SOS_IDX, EOS_IDX).to(device)
else:
    # Transformer model: same forward(src, src_lens, tgt) / translate(...) API as
    # the RNN Seq2Seq, so the training loop and evaluation below are unchanged.
    model = TransformerSeq2Seq(
        INPUT_DIM, OUTPUT_DIM, device,
        d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
        d_ff=D_FF, dropout=TF_DROPOUT,
        pad_idx=pad_idx, sos_idx=SOS_IDX, eos_idx=EOS_IDX,
    ).to(device)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

The model has 36,553,428 trainable parameters


## 4. Training Loop
Define the functions for training (with Teacher Forcing) and evaluating (without Teacher Forcing) for a single epoch.

In [7]:
from torch.amp import autocast

USE_AMP = device.type == 'cuda' and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16
print(f'AMP enabled: {USE_AMP} (dtype: {AMP_DTYPE if USE_AMP else "fp32"})')

def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    for batch in iterator:
        src, src_lens, tgt, _ = batch
        src, src_lens, tgt = src.to(device), src_lens.to(device), tgt.to(device)

        optimizer.zero_grad()

        with autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            output = model(src, src_lens, tgt)
            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            tgt_out = tgt[:, 1:].reshape(-1)
            loss = criterion(output, tgt_out)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()
    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for _, batch in enumerate(iterator):
            src, src_lens, tgt, _ = batch
            src, src_lens, tgt = src.to(device), src_lens.to(device), tgt.to(device)

            # Turn off teacher forcing for evaluation (ratio = 0)
            output = model(src, src_lens, tgt, teacher_forcing_ratio=0)

            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            tgt = tgt[:, 1:].reshape(-1)

            loss = criterion(output, tgt)
            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

AMP enabled: True (dtype: torch.bfloat16)


Execute training

In [8]:
best_valid_loss = float('inf')
if MODEL_TYPE == 'RNN':
    model_filename = f"rnn_emb{ENC_EMB_DIM}_hid{ENC_HID_DIM}_lr{LEARNING_RATE}.pt"
else:
    model_filename = f"transformer_d{D_MODEL}_h{N_HEADS}_l{N_LAYERS}_lr{LEARNING_RATE}.pt"
model_save_path = os.path.join(models_dir, model_filename)
print(f"Starting training... Model will be saved to: {model_save_path}")

for epoch in range(N_EPOCHS):
    start_time = time.time()
    
    train_loss = train(model, train_loader, optimizer, criterion, CLIP)
    valid_loss = evaluate(model, val_loader, criterion)
    scheduler.step(valid_loss)
    
    end_time = time.time()
    
    # Save the model if it's the best one we've seen so far
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), model_save_path)
    
    # Perplexity (PPL) is a standard metric in NLP, calculated as exp(loss)
    print(f'Epoch: {epoch+1:02} | Time: {int(end_time - start_time)}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')

Starting training... Model will be saved to: /home/jovyan/lectures/192-183-2026s/assignments/910/Machine_Translation-main/models/transformer_d256_h8_l3_lr0.0005.pt


/opt/micromamba/lib/python3.11/site-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch: 01 | Time: 153s
	Train Loss: 3.206 | Train PPL:  24.674
	 Val. Loss: 2.287 |  Val. PPL:   9.841
Epoch: 02 | Time: 151s
	Train Loss: 2.102 | Train PPL:   8.186
	 Val. Loss: 1.948 |  Val. PPL:   7.012
Epoch: 03 | Time: 150s
	Train Loss: 1.755 | Train PPL:   5.781
	 Val. Loss: 1.815 |  Val. PPL:   6.143
Epoch: 04 | Time: 154s
	Train Loss: 1.552 | Train PPL:   4.719
	 Val. Loss: 1.735 |  Val. PPL:   5.668
Epoch: 05 | Time: 150s
	Train Loss: 1.410 | Train PPL:   4.095
	 Val. Loss: 1.707 |  Val. PPL:   5.510
Epoch: 06 | Time: 151s
	Train Loss: 1.311 | Train PPL:   3.711
	 Val. Loss: 1.688 |  Val. PPL:   5.408
Epoch: 07 | Time: 150s
	Train Loss: 1.235 | Train PPL:   3.438
	 Val. Loss: 1.664 |  Val. PPL:   5.281
Epoch: 08 | Time: 150s
	Train Loss: 1.181 | Train PPL:   3.256
	 Val. Loss: 1.641 |  Val. PPL:   5.163
Epoch: 09 | Time: 151s
	Train Loss: 1.130 | Train PPL:   3.094
	 Val. Loss: 1.618 |  Val. PPL:   5.043
Epoch: 10 | Time: 150s
	Train Loss: 1.064 | Train PPL:   2.897
	 Val. Los

## 5. Inference and Evaluation
Load the best model and translate sentences from the test set.

In [9]:
# Load best weights
model.load_state_dict(torch.load(model_save_path))

def strip_special(tokens):
    return [t for t in tokens if t not in (Vocabulary.PAD, Vocabulary.SOS, Vocabulary.EOS, Vocabulary.UNK)]

# Translate a few sentences from the test set
print("\n--- Example Translations ---")
model.eval()
with torch.no_grad():
    batch = next(iter(test_loader))
    src, src_lens, tgt, _ = batch
    src, src_lens, tgt = src.to(device), src_lens.to(device), tgt.to(device)
    
    # Use the greedy translation method implemented in seq2seq.py
    # TODO (Optional): Implement Beam Search in seq2seq.py and use it here instead of greedy decoding
    translations, attentions = model.translate(src, src_lens, max_len=50)
    
    for i in range(min(5, src.size(0))): # Show up to 5 examples
        src_sentence = ' '.join(strip_special(src_vocab.decode(src[i].tolist())))
        tgt_sentence = ' '.join(strip_special(tgt_vocab.decode(tgt[i].tolist())))
        pred_sentence = ' '.join(strip_special(tgt_vocab.decode(translations[i].tolist())))
        
        print(f"\nSource: {src_sentence}")
        print(f"Target: {tgt_sentence}")
        print(f"Predicted: {pred_sentence}")


--- Example Translations ---

Source: it's not for everybody .
Target: ce n'est pas tout public .
Predicted: ce n'est pas pour tout le monde .

Source: this is not a fish .
Target: ce n'est pas un poisson .
Predicted: ce n'est pas un poisson .

Source: i rode a horse for the first time yesterday .
Target: je suis monté à cheval pour la première fois hier .
Predicted: j'ai monté un cheval pour la première fois hier .

Source: sixty percent of japanese adult males drink alcoholic beverages on a regular basis .
Target: soixante pourcents des hommes adultes japonais boivent régulièrement de l'alcool .
Predicted: les japonais consomment de l'alcool japonais consomment des alcoolique traditionnelle japonaise à une alcoolique traditionnelle japonaise .

Source: we have people who do that for us .
Target: nous avons des gens qui font cela pour nous .
Predicted: nous avons des gens qui font ça pour nous .


In [10]:
from nltk.translate.bleu_score import corpus_bleu

def compute_bleu(model, loader, src_vocab, tgt_vocab, device, max_len=50):
    model.eval()
    hypotheses = []   # list of list of tokens
    references  = []  # list of list of list of tokens (corpus_bleu format)
    
    special = {Vocabulary.PAD, Vocabulary.SOS, Vocabulary.EOS, Vocabulary.UNK}
    
    with torch.no_grad():
        for src, src_lens, tgt, _ in loader:
            src, src_lens = src.to(device), src_lens.to(device)
            translations, _ = model.translate(src, src_lens, max_len=max_len)
            
            for i in range(src.size(0)):
                pred = [t for t in tgt_vocab.decode(translations[i].tolist())
                        if t not in special]
                ref  = [t for t in tgt_vocab.decode(tgt[i].tolist())
                        if t not in special]
                hypotheses.append(pred)
                references.append([ref])   # corpus_bleu attend [[ref1], [ref2], ...]
    
    return corpus_bleu(references, hypotheses) * 100   # score en %

bleu = compute_bleu(model, test_loader, src_vocab, tgt_vocab, device)
print(f"BLEU score on test set: {bleu:.2f}")

BLEU score on test set: 34.48


In [ ]:
# COMET score calculation
import os
os.environ["USE_TF"] = "0"

from comet import download_model, load_from_checkpoint
def compute_comet(model, loader, src_vocab, tgt_vocab, device, max_len=50):
    """
    Compute COMET score on a test set.
    import torch
import transformers
import google.protobuf

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Protobuf:", google.protobuf.__version__)
    COMET (Crosslingual Optimized Metric for Evaluation of Translation) is a neural 
    metric that correlates better with human judgment than BLEU.
    
    Args:
        model: the NMT model (Seq2Seq or Transformer)
        loader: DataLoader with test data
        src_vocab: source vocabulary
        tgt_vocab: target vocabulary
        device: torch device
        max_len: maximum translation length
        
    Returns:
        Average COMET score (0-1 range)
    """
    model.eval()
    sources = []       # list of source sentences (strings)
    hypotheses = []    # list of translated sentences (strings)
    references = []    # list of reference sentences (strings)
    
    special = {Vocabulary.PAD, Vocabulary.SOS, Vocabulary.EOS, Vocabulary.UNK}
    
    # Generate translations for all test examples
    with torch.no_grad():
        for src, src_lens, tgt, _ in loader:
            src, src_lens = src.to(device), src_lens.to(device)
            translations, _ = model.translate(src, src_lens, max_len=max_len)
            
            for i in range(src.size(0)):
                # Decode and clean each sequence
                src_tokens = [t for t in src_vocab.decode(src[i].tolist()) if t not in special]
                hyp_tokens = [t for t in tgt_vocab.decode(translations[i].tolist()) if t not in special]
                ref_tokens = [t for t in tgt_vocab.decode(tgt[i].tolist()) if t not in special]
                
                # Convert to strings
                sources.append(' '.join(src_tokens))
                hypotheses.append(' '.join(hyp_tokens))
                references.append(' '.join(ref_tokens))
    
    # Load pre-trained COMET model
    print("Downloading COMET model...")
    comet_model = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))
    
    # Prepare data in the format expected by COMET
    data = [
        {"src": src, "mt": hyp, "ref": ref}
        for src, hyp, ref in zip(sources, hypotheses, references)
    ]
    
    # Compute COMET scores
    print("Computing COMET scores...")
    scores = comet_model.predict(data, batch_size=8, gpus=1 if device.type == 'cuda' else 0)
    
    # Return average system score (0-1 range, typically higher is better)
    return scores.system_score * 100  # scale to 0-100 for easier interpretation

# Compute and display COMET score
comet = compute_comet(model, test_loader, src_vocab, tgt_vocab, device)
print(f"COMET score on test set: {comet:.2f}")